In [3]:
loan_data = [
(101,1,50000,"Home","Approved","2025-01-01","2025-01-05"),
(102,2,20000,"Personal","Pending","2025-01-02",None),
(103,3,30000,"Auto","Approved","2025-01-03","2025-01-04"),
(104,4,45000,"Education","Rejected","2025-01-04","2025-01-02"),
(105,5,70000,"Home","Approved","2025-01-05","2025-01-10"),
(106,6,15000,"Personal","Approved","2025-01-06","2025-01-08"),
(107,7,80000,"Auto","Pending","2025-01-07",None),
(108,8,12000,"Education","Approved","2025-01-08","2025-01-09"),
(109,9,60000,"Home","Approved","2025-01-09","2025-01-12"),
(110,10,25000,"Personal","Rejected","2025-01-10","2025-01-11")
]
loan_data_columns = ["LoanID","CustomerID","LoanAmount","LoanType","LoanStatus","LoanDate","ApprovalDate"]



In [4]:
%%writefile loan.csv
"LoanID","CustomerID","LoanAmount","LoanType","LoanStatus","LoanDate","ApprovalDate"
101,1,50000,"Home","Approved","2025-01-01","2025-01-05"
102,2,20000,"Personal","Pending","2025-01-02",None
103,3,30000,"Auto","Approved","2025-01-03","2025-01-04"
104,4,45000,"Education","Rejected","2025-01-04","2025-01-02"
105,5,70000,"Home","Approved","2025-01-05","2025-01-10"
106,6,15000,"Personal","Approved","2025-01-06","2025-01-08"
107,7,80000,"Auto","Pending","2025-01-07",None
108,8,12000,"Education","Approved","2025-01-08","2025-01-09"
109,9,60000,"Home","Approved","2025-01-09","2025-01-12"
110,10,25000,"Personal","Rejected","2025-01-10","2025-01-11"

Overwriting loan.csv


In [5]:
customer_data = [
(1,"Rahul","Sharma","Mumbai"),
(2,"Priya",None,"Pune"),
(3,"Aman","Verma","Delhi"),
(4,"Sneha","Iyer","Chennai"),
(5,"Arjun","Reddy","Hyderabad"),
(6,"Kavya","Nair","Bangalore"),
(7,"Rohit","Das","Kolkata"),
(8,"Meera",None,"Jaipur"),
(9,"Karan","Patel","Surat"),
(10,"Neha","Joshi","Ahmedabad")
]

customer_data_columns = ["CustomerID","FirstName","LastName","City"]


In [6]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql import functions as F

class BankLoanAnalyzer:
    def __init__(self):
        pass
        
    def create_loan_df(self, spark: SparkSession) -> DataFrame:
        df = spark.createDataFrame(loan_data,loan_data_columns)
        df=df.withColumn("LoanDate",F.to_date("LoanDate"))
        df=df.withColumn("ApprovalDate",F.to_date("ApprovalDate"))
        return df
        
    def create_customer_df(self, spark: SparkSession) -> DataFrame:
        return spark.createDataFrame(customer_data,customer_data_columns)

    def compute_processing_days(self, df: DataFrame) -> DataFrame:
        df=df.withColumn("ProcessingDays",F.datediff(F.col("ApprovalDate"), F.col("LoanDate")))
        return df

    def filter_valid_loans(self, df: DataFrame) -> DataFrame:
        return df.filter(F.col("ProcessingDays") >= 0)

    def join_customer_data(self, loan_df: DataFrame, customer_df: DataFrame) -> DataFrame:
        return loan_df.join(customer_df,on="CustomerID",how="left").select("LoanID","CustomerID","FirstName","LastName","City","LoanAmount").orderBy("LoanID","CustomerID")

    def enrich_full_name(self, df: DataFrame) -> DataFrame:
        return df.withColumn("FullName",F.concat_ws(" ", F.col("FirstName"), F.col("LastName"))).select("CustomerID","FullName")

    def loan_summary(self, df: DataFrame) -> DataFrame:
        return df.groupby("LoanStatus").agg(F.sum("LoanAmount").alias("sum"),F.avg("LoanAmount").alias("avg"),F.min("LoanAmount").alias("min"),F.max("LoanAmount").alias("max"))

    def top_loan_type(self, df: DataFrame) -> tuple:
        return tuple(df.groupby("LoanType").count().orderBy(F.col("count").desc(),F.col("LoanType")).collect()[0])

    def unique_loan_types(self, df: DataFrame) -> list:
        return df.select("LoanType").distinct().rdd.flatMap(lambda x:x).collect()

    def high_value_loans(self, df: DataFrame, threshold: int) -> DataFrame:
        return df.filter(F.col("LoanAmount") > threshold).select("LoanID","LoanAmount")

    def avg_processing_by_type(self, df: DataFrame) -> DataFrame:
        return df.groupby("LoanType").agg(F.avg("ProcessingDays").alias("AvgDays"))

    def top_customers(self, df: DataFrame, n: int) -> DataFrame:
        return df.groupby("CustomerID").agg(F.sum("LoanAmount").alias("Total")).orderBy(F.col("Total").desc()).limit(n)

    def define_loan_schema(self) -> StructType:
        return StructType([StructField("LoanID",IntegerType(),True),
                        StructField("CustomerID",IntegerType(),True),
                        StructField("LoanAmount",IntegerType(),True),
                        StructField("LoanType",StringType(),True),
                        StructField("LoanStatus",StringType(),True),
                        StructField("LoanDate",StringType(),True),
                        StructField("ApprovalDate",StringType(),True)])
        
    def load_loan_csv(self, spark: SparkSession, path: str, schema: StructType) -> DataFrame:
        return spark.read.csv(path,header=True,schema=schema)

    def filter_by_date_range(self, df: DataFrame, start: str, end: str) -> DataFrame:
        return df.filter(F.col("LoanDate").between(start,end))

    def add_month_column(self, df: DataFrame) -> DataFrame:
        df=df.withColumn("Month",F.date_trunc("month", F.col("LoanDate")).cast("date"))
        return df

    def apply_risk_category(self, df: DataFrame) -> DataFrame:
        return df.withColumn("RiskCategory",F.when(F.col("LoanAmount")==60000,"High").when((F.col("LoanAmount")>=30000)& (F.col("LoanAmount")<30000), "Medium").otherwise("Low"))

    def monthly_loan_summary(self, df: DataFrame) -> DataFrame:
        return df.groupby("LoanType","Month").agg(F.sum("LoanAmount").alias("TotalLoan"))

    def get_highest_loan(self, df: DataFrame) -> tuple:
        return tuple(df.orderBy(F.col("LoanAmount").desc()).select("LoanID","LoanAmount").first())

    def avg_loan_per_customer(self, df: DataFrame) -> DataFrame:
        return df.groupby("CustomerID").agg(F.avg("LoanAmount").alias("AvgLoan"))        

In [7]:
spark=SparkSession.builder.appName("").getOrCreate()

b = BankLoanAnalyzer()
loan_df=b.create_loan_df(spark)
loan_df.show()

customer_df=b.create_customer_df(spark)
customer_df.show()

loan_df1=b.compute_processing_days(loan_df)
loan_df1.show()

filtered_df=b.filter_valid_loans(loan_df1)
filtered_df.show()

df4 = b.join_customer_data(loan_df, customer_df)
df4.show()

df5=b.enrich_full_name(customer_df)
df5.show()

df6=b.loan_summary(loan_df)
df6.show()

tuple1=b.top_loan_type(loan_df)
print(tuple1)

list1=b.unique_loan_types(loan_df)
print(list1)

df7=b.high_value_loans(loan_df, 50000)
df7.show()

df8=b.avg_processing_by_type(loan_df1)
df8.show()

df9=b.top_customers(loan_df1, 2)
df9.show()

schema=b.define_loan_schema()

df10 = b.load_loan_csv(spark, "loan.csv", schema)
df10.show()

df11 = b.filter_by_date_range(df10, "2025-01-01", "2025-01-03")
df11.show()

df12=b.add_month_column(df10)
df12.show()

df13=b.apply_risk_category(df12)
df13.show()

df14=b.monthly_loan_summary(df13)
df14.show()

tuple2=b.get_highest_loan(df13)
print(tuple2)

df15=b.avg_loan_per_customer(df13)
df15.show()

+------+----------+----------+---------+----------+----------+------------+
|LoanID|CustomerID|LoanAmount| LoanType|LoanStatus|  LoanDate|ApprovalDate|
+------+----------+----------+---------+----------+----------+------------+
|   101|         1|     50000|     Home|  Approved|2025-01-01|  2025-01-05|
|   102|         2|     20000| Personal|   Pending|2025-01-02|        NULL|
|   103|         3|     30000|     Auto|  Approved|2025-01-03|  2025-01-04|
|   104|         4|     45000|Education|  Rejected|2025-01-04|  2025-01-02|
|   105|         5|     70000|     Home|  Approved|2025-01-05|  2025-01-10|
|   106|         6|     15000| Personal|  Approved|2025-01-06|  2025-01-08|
|   107|         7|     80000|     Auto|   Pending|2025-01-07|        NULL|
|   108|         8|     12000|Education|  Approved|2025-01-08|  2025-01-09|
|   109|         9|     60000|     Home|  Approved|2025-01-09|  2025-01-12|
|   110|        10|     25000| Personal|  Rejected|2025-01-10|  2025-01-11|
+------+----

+----------+---------+--------+---------+
|CustomerID|FirstName|LastName|     City|
+----------+---------+--------+---------+
|         1|    Rahul|  Sharma|   Mumbai|
|         2|    Priya|    NULL|     Pune|
|         3|     Aman|   Verma|    Delhi|
|         4|    Sneha|    Iyer|  Chennai|
|         5|    Arjun|   Reddy|Hyderabad|
|         6|    Kavya|    Nair|Bangalore|
|         7|    Rohit|     Das|  Kolkata|
|         8|    Meera|    NULL|   Jaipur|
|         9|    Karan|   Patel|    Surat|
|        10|     Neha|   Joshi|Ahmedabad|
+----------+---------+--------+---------+

+------+----------+----------+---------+----------+----------+------------+--------------+
|LoanID|CustomerID|LoanAmount| LoanType|LoanStatus|  LoanDate|ApprovalDate|ProcessingDays|
+------+----------+----------+---------+----------+----------+------------+--------------+
|   101|         1|     50000|     Home|  Approved|2025-01-01|  2025-01-05|             4|
|   102|         2|     20000| Personal|   Pend

+------+----------+---------+--------+---------+----------+
|LoanID|CustomerID|FirstName|LastName|     City|LoanAmount|
+------+----------+---------+--------+---------+----------+
|   101|         1|    Rahul|  Sharma|   Mumbai|     50000|
|   102|         2|    Priya|    NULL|     Pune|     20000|
|   103|         3|     Aman|   Verma|    Delhi|     30000|
|   104|         4|    Sneha|    Iyer|  Chennai|     45000|
|   105|         5|    Arjun|   Reddy|Hyderabad|     70000|
|   106|         6|    Kavya|    Nair|Bangalore|     15000|
|   107|         7|    Rohit|     Das|  Kolkata|     80000|
|   108|         8|    Meera|    NULL|   Jaipur|     12000|
|   109|         9|    Karan|   Patel|    Surat|     60000|
|   110|        10|     Neha|   Joshi|Ahmedabad|     25000|
+------+----------+---------+--------+---------+----------+

+----------+------------+
|CustomerID|    FullName|
+----------+------------+
|         1|Rahul Sharma|
|         2|       Priya|
|         3|  Aman Verma|
|  